## **1. Config and Imports**

- **General imports**

In [11]:
import json
import re
from datetime import datetime, timezone
from xml.etree import ElementTree as ET
import httpx

- **Load environment variable values**

In [12]:
import os 
from dotenv import load_dotenv

# Load the environment variables from the .env file
load_dotenv()


# Acess the .env variable values
NCBI_EMAIL = os.getenv("NCBI_EMAIL")
NCBI_API_KEY = os.getenv("NCBI_API_KEY")
OPENALEX_API_KEY = os.getenv("OPENALEX_API_KEY")

if not NCBI_EMAIL:
    raise RuntimeError("Defina NCBI_EMAIL no arquivo .env")

In [13]:
from pathlib import Path

current_dir = Path.cwd()
while not (current_dir / "pyproject.toml").exists():
    if current_dir == current_dir.parent:
        raise RuntimeError("pyproject.toml not found")
    current_dir = current_dir.parent

PROJECT_ROOT = current_dir

- **Build directories**

In [14]:
RAW_PUBMED_DIR = PROJECT_ROOT / "data" / "raw" / "pubmed"
RAW_CROSSREF_DIR = PROJECT_ROOT / "data" / "raw" / "crossref"
RAW_OPENALEX_DIR = PROJECT_ROOT / "data" / "raw" / "openalex"
DB_PATH = PROJECT_ROOT / "data" / "interim" / "bibliometria.db"

for directory in [
    RAW_PUBMED_DIR,
    RAW_CROSSREF_DIR,
    RAW_OPENALEX_DIR,
    DB_PATH.parent,
]:
    directory.mkdir(parents=True, exist_ok=True)

- **Run config**

In [15]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
SAMPLE_SIZE = 20

print("Project root:", PROJECT_ROOT)
print("Database:", DB_PATH)
print("Execution ID:", RUN_ID)

Project root: /home/lucas/github/bibliometric-rag-assistant
Database: /home/lucas/github/bibliometric-rag-assistant/data/interim/bibliometria.db
Execution ID: 20260908T152213Z


## **2. Building Query**

- **Indexação padronizada [Mesh] ou linguagem empregada por diferentes autores (título, abstract e palavras-chave)**

In [9]:
QUERY = """
(
    "Temporomandibular Joint"[Mesh]                            
    OR "temporomandibular joint"[Title/Abstract]               
)
AND
(
    "Joint Prosthesis"[Mesh]
    OR prosthe*[Title/Abstract]
    OR replacement[Title/Abstract]
    OR alloplastic[Title/Abstract]
)
AND
(
    infection*[Title/Abstract]
    OR antibiotic*[Title/Abstract]
    OR prophylaxis[Title/Abstract]
)
AND
(
    "2000/01/01"[Date - Publication]
    :
    "2025/12/31"[Date - Publication]
)
"""

QUERY = " ".join(QUERY.split())

print(QUERY)

( "Temporomandibular Joint"[Mesh] OR "temporomandibular joint"[Title/Abstract] ) AND ( "Joint Prosthesis"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( "2000/01/01"[Date - Publication] : "2025/12/31"[Date - Publication] )


In [23]:
QUERY_ARTICLE_LITERAL = """
(
    "temporomandibular joint"
    OR "temporomandibular replacement"
    OR TMJ
    OR TJR
    OR ETJA
)
AND
(
    prosthesis
    OR replacement
)
AND
(
    prevention
    OR prophylaxis
    OR "infection control"
    OR antibiotics
    OR detection
)
AND
(
    "2000/01/01"[Date - Publication]
    :
    "2025/12/31"[Date - Publication]
)
"""

QUERY_ARTICLE_LITERAL = " ".join(
    QUERY_ARTICLE_LITERAL.split()
)

In [24]:
QUERY_ARTICLE_EXPLICIT = """
(
    "Temporomandibular Joint"[Mesh]
    OR "temporomandibular joint"[Title/Abstract]
    OR "temporomandibular replacement"[Title/Abstract]
    OR TMJ[Title/Abstract]
    OR TJR[Title/Abstract]
    OR ETJA[Title/Abstract]
)
AND
(
    prosthesis[Title/Abstract]
    OR replacement[Title/Abstract]
)
AND
(
    prevention[Title/Abstract]
    OR prophylaxis[Title/Abstract]
    OR "infection control"[Title/Abstract]
    OR antibiotics[Title/Abstract]
    OR detection[Title/Abstract]
)
AND
(
    "2000/01/01"[Date - Publication]
    :
    "2025/12/31"[Date - Publication]
)
"""

QUERY_ARTICLE_EXPLICIT = " ".join(
    QUERY_ARTICLE_EXPLICIT.split()
)

## **3. Sending a Query to the PubMed**

In [ ]:
from bibliometric_rag_assistant.search_strategy import (
    PUBMED_STATUS, PUBMED_PERIOD, PUBMED_SAMPLE_SIZE,
    PUBMED_ACTIVE_VARIANT, PUBMED_STRATEGY, PUBMED_VARIANTS, 
    PUBMED_QUERY, PUBMED_SOURCE
)


In [7]:
EUTILS_BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

client = httpx.Client(
    timeout=30,
    follow_redirects=True,
    headers={"User-Agent": f"bibliometric-rag-assistant/0.1 ({NCBI_EMAIL})"},
)


def search_pubmed(query: str = PUBMED_QUERY, limit: int = PUBMED_SAMPLE_SIZE) -> dict:
    params = {
        "db": PUBMED_SOURCE,
        "term": query,
        "retmode": "json", 
        "retmax": limit,
        "tool": "bibliometric_rag_assistant",
        "email": NCBI_EMAIL,
    }

    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY

    response = client.get(
        f"{EUTILS_BASE_URL}/esearch.fcgi",
        params=params,
    )
    response.raise_for_status() # HTTP object

    return response.json() # json -> dictionary


search_result = search_pubmed()

pmids = search_result["esearchresult"]["idlist"]
total_found = int(search_result["esearchresult"]["count"])

search_path = RAW_PUBMED_DIR / f"{RUN_ID}_esearch.json"
search_path.write_text(
    json.dumps(search_result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Total encontrado pelo PubMed:", total_found)
print("PMIDs recuperados na amostra:", len(pmids))
print("Primeiros PMIDs:", pmids[:5])

Total encontrado pelo PubMed: 100
PMIDs recuperados na amostra: 20
Primeiros PMIDs: ['41328123', '41076314', '40725551', '40393895', '40335417']


In [35]:
queries_to_compare = {
    "descrita_literalmente": QUERY_ARTICLE_LITERAL,
    "descrita_com_campos": QUERY_ARTICLE_EXPLICIT,
    "query_provisoria_atual": QUERY,
}

for name, query_text in queries_to_compare.items():
    result = search_pubmed(query_text, limit=0)
    esearch = result["esearchresult"]

    print("\nQuery:", name)
    print("Total:", esearch["count"])
    print(
        "Interpretação do PubMed:",
        esearch.get("querytranslation"),
    )
    print("Avisos:", esearch.get("warninglist"))
    print("Erros:", esearch.get("errorlist"))


Query: descrita_literalmente
Total: 336
Interpretação do PubMed: ("temporomandibular joint"[All Fields] OR "temporomandibular replacement"[All Fields] OR ("temporomandibular joint"[MeSH Terms] OR ("temporomandibular"[All Fields] AND "joint"[All Fields]) OR "temporomandibular joint"[All Fields] OR "tmj"[All Fields]) OR "TJR"[All Fields] OR "ETJA"[All Fields]) AND ("prostheses and implants"[MeSH Terms] OR ("prostheses"[All Fields] AND "implants"[All Fields]) OR "prostheses and implants"[All Fields] OR "prosthesis"[All Fields] OR ("replace"[All Fields] OR "replaceable"[All Fields] OR "replaced"[All Fields] OR "replaces"[All Fields] OR "replacing"[All Fields] OR "replacment"[All Fields] OR "replantation"[MeSH Terms] OR "replantation"[All Fields] OR "replacement"[All Fields] OR "replacements"[All Fields])) AND ("prevent"[All Fields] OR "preventability"[All Fields] OR "preventable"[All Fields] OR "preventative"[All Fields] OR "preventatively"[All Fields] OR "preventatives"[All Fields] OR "p

## **4. Returning data from pmids**

In [32]:
def fetch_pubmed_articles(pmids: list[str]) -> str:
    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml",
        "tool": "bibliometric_rag_assistant",
        "email": NCBI_EMAIL,
    }

    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY

    response = client.get(
        f"{EUTILS_BASE_URL}/efetch.fcgi",
        params=params,
    )
    response.raise_for_status()

    return response.text


pubmed_xml = fetch_pubmed_articles(pmids)

fetch_path = RAW_PUBMED_DIR / f"{RUN_ID}_efetch.xml"
fetch_path.write_text(pubmed_xml, encoding="utf-8")

print("XML salvo em:", fetch_path)
print("Tamanho:", len(pubmed_xml), "caracteres")

XML salvo em: /home/lucas/github/bibliometric-rag-assistant/data/raw/pubmed/20260904T111733Z_efetch.xml
Tamanho: 253969 caracteres


- **Parsing**

In [33]:
def get_element_text(element) -> str | None:
    if element is None:
        return None

    text = "".join(element.itertext()).strip()
    return text or None


def extract_publication_year(article_node) -> int | None:
    year = article_node.findtext(
        "Journal/JournalIssue/PubDate/Year"
    )

    if year:
        return int(year)

    medline_date = article_node.findtext(
        "Journal/JournalIssue/PubDate/MedlineDate"
    )

    if medline_date:
        match = re.search(r"\b(19|20)\d{2}\b", medline_date)

        if match:
            return int(match.group())

    return None


def parse_pubmed_xml(xml_content: str) -> list[dict]:
    root = ET.fromstring(xml_content)
    records = []

    for pubmed_article in root.findall("PubmedArticle"):
        citation = pubmed_article.find("MedlineCitation")
        article = citation.find("Article")

        pmid = citation.findtext("PMID")
        title = get_element_text(article.find("ArticleTitle"))

        abstract_parts = []

        for abstract_element in article.findall(
            "Abstract/AbstractText"
        ):
            text = get_element_text(abstract_element)

            if not text:
                continue

            label = abstract_element.attrib.get("Label")

            if label:
                abstract_parts.append(f"{label}: {text}")
            else:
                abstract_parts.append(text)

        authors = []

        for author in article.findall("AuthorList/Author"):
            collective_name = author.findtext("CollectiveName")

            if collective_name:
                authors.append(collective_name)
                continue

            first_name = author.findtext("ForeName") or ""
            last_name = author.findtext("LastName") or ""

            full_name = f"{first_name} {last_name}".strip()

            if full_name:
                authors.append(full_name)

        affiliations = sorted(
            {
                affiliation.text.strip()
                for affiliation in article.findall(
                    "AuthorList/Author/AffiliationInfo/Affiliation"
                )
                if affiliation.text
            }
        )

        doi = None

        for article_id in pubmed_article.findall(
            "PubmedData/ArticleIdList/ArticleId"
        ):
            if article_id.attrib.get("IdType") == "doi":
                doi = article_id.text
                break

        records.append(
            {
                "pmid": pmid,
                "doi": doi,
                "title": title,
                "abstract": "\n".join(abstract_parts) or None,
                "journal": article.findtext("Journal/Title"),
                "publication_year": extract_publication_year(article),
                "language": article.findtext("Language"),
                "authors": authors,
                "affiliations": affiliations,
            }
        )

    return records


articles = parse_pubmed_xml(pubmed_xml)

print("Artigos processados:", len(articles))
articles[0]

Artigos processados: 20


{'pmid': '41328123',
 'doi': '10.7759/cureus.95806',
 'title': 'Evolving Horizons in Temporomandibular Joint Total Replacement: A Comprehensive Narrative Review.',
 'abstract': 'Total temporomandibular joint (TMJ) replacement has become a vital solution for managing end-stage TMJ disorders and significantly enhancing mandibular function and patient quality of life. This narrative review explores the historical developments, clinical objectives, biomaterials, devices, surgical techniques, preoperative preparation, complications, management approaches, and recent innovations in TMJ total joint replacement (TJR). The review draws from a literature search conducted across PubMed, Scopus, Web of Science, and Google Scholar from database inception to August 10, 2025, using terms like "total temporomandibular joint replacement," "TMJ TJR," and "alloplastic TMJ prosthesis," focusing on English-language peer-reviewed articles, reviews, and clinical studies. Current practice leans toward allopla